In [ ]:
#分类和定位

#添加具有四个单位的第二个密集输出（通常在全局平均池化层之上），就可以使用MSE损失对其进行训练

base_model = keras.applications.xception.Xception(weights="imagenet", include_top=False) 
avg = keras.layers.GlobalAveragePooling2D()(base_model.output) 
class_output = keras.layers.Dense(n_classes, activation="softmax")(avg) 
loc_output = keras.layers.Dense(4)(avg) 
model = keras.Model(inputs=base_model.input, outputs=[class_output, loc_output]) 
model.compile(loss=["sparse_categorical_crossentropy", "mse"], 
              loss_weights=[0.8, 0.2], # depends on what you care most about 
              optimizer=optimizer, metrics=["accuracy"])

MSE通常作为成本函数可以很好地训练模型，但是评估模型对边界
框的预测能力不是一个很好的指标。最常用的度量指标是“交并比”
（Intersection over Union，IoU）：预测边界框和目标边界框之间的
重叠面积除以它们的联合面积

In [ ]:
#物体检测
#1.首先你需要在CNN中添加一个额外的客观分数（置信度）输出，以估计图像中确实存在花朵的可能性
#2.找到具有最客观分数的边界框，并删除与其重叠很多的所有其他边界框
#3.重复第二步，知道没有更多的边界框可以删除